In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session



/kaggle/input/ae4353-y25/LittletonHQ.h5
/kaggle/input/ae4353-y25/Warsaw1.h5
/kaggle/input/ae4353-y25/autonomous_flight-16a-trackRATM.h5
/kaggle/input/ae4353-y25/sample_submission.csv
/kaggle/input/ae4353-y25/piloted_flight-03p-ellipse.h5
/kaggle/input/ae4353-y25/BaltimoreOB.h5
/kaggle/input/ae4353-y25/autonomous_flight-10a-lemniscate.h5
/kaggle/input/ae4353-y25/piloted_flight-08p-lemniscate.h5
/kaggle/input/ae4353-y25/autonomous_flight-03a-ellipse.h5
/kaggle/input/ae4353-y25/LittletonBlue.h5
/kaggle/input/ae4353-y25/autonomous_flight-15a-trackRATM.h5
/kaggle/input/ae4353-y25/piloted_flight-09p-lemniscate.h5
/kaggle/input/ae4353-y25/piloted_flight-05p-ellipse.h5
/kaggle/input/ae4353-y25/autonomous_flight-17a-trackRATM.h5
/kaggle/input/ae4353-y25/piloted_flight-06p-ellipse.h5
/kaggle/input/ae4353-y25/autonomous_flight-04a-ellipse.h5
/kaggle/input/ae4353-y25/autonomous_flight-14a-trackRATM.h5
/kaggle/input/ae4353-y25/piloted_flight-11p-lemniscate.h5
/kaggle/input/ae4353-y25/test_set.h5
/k

In [2]:
!pip install hdf5plugin opencv-python-headless moviepy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 34.9 MB/s eta 0:00:00


In [3]:
import h5py
import hdf5plugin

import matplotlib.pyplot as plt
from moviepy.editor import ImageSequenceClip

import cv2
from pathlib import Path

import torch
from torchvision import transforms
from torch.utils.data import DataLoader, ConcatDataset, Subset, Dataset
from tqdm import tqdm
import torch.nn.functional as F
from torch import nn

error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evalu

In [4]:
# dataset = '/kaggle/input/ae4353-y25/BaltimoreMobile.h5'
# with h5py.File('/kaggle/input/ae4353-y25/LittletonBlue.h5', 'r') as f:

#     # simplifications made to the dataset
#     # - only take first gate into account, not multiple gates per picture
#     # - training on non all visible gate coordinates is possible
    
#     images = f['images']    # list of images, can be indexed by [i], and can be multiple gates per one image
#     targets = f['targets']  # list of target coordinates indicated by ['i'], and can be multiple gates coordinates per one image
    
#     # Matplotlib expects image shape (H, W, C), but we have (C, H, W).
#     # We need to transpose the axes.
#     image_for_plot = np.transpose(images, (1, 2, 0))
    
    # # --- Plotting ---
    # plt.figure(figsize=(10, 8))
    # plt.imshow(image_for_plot)
    # plt.title("Full Image Display")
    
    # # --- ADD THIS LINE ---
    # plt.axis('off') 
    
    # plt.show()

In [5]:
class H5DroneDataset(Dataset):
    """
    Handles variable numbers of gates by padding targets to a uniform length
    and generating a custom mask for each sample.
    """
    def __init__(self, h5_paths, transform=None):
        
        self.transform = transform
        self.max_gates = 5

        all_clean_images = []
        all_clean_targets = []

        print("Scanning all files to filter out multi gate images...")
        
        # Scan all files to find the max number of gates and collect clean data
        for h5_path in tqdm(h5_paths, desc="Scanning Files"):
            
            with h5py.File(h5_path, 'r') as f:
                images_data = f['images'][:]
                
                keys = sorted(f['targets'].keys(), key=int)
                targets_data = [f['targets'][key][:] for key in keys]

                # Your filtering logic is now correct!
                for idx, target in enumerate(targets_data):
                    if target.size == 12: # A cleaner way to write the check
                        all_clean_images.append(images_data[idx])
                        all_clean_targets.append(target)
                    
                    # # Loop for if multiple images, but now only train with images that are one gate.
                    # if target.size > 0 and target.shape[-1] == 12:
                    #     gates = target.shape[1]

                    #     if gates > 1 and gates < self.max_gates:  # NOTE: Changed the condition to check if padding is actually needed
                    #         # 1. Calculate how many zero rows are needed
                    #         padding_needed = self.max_gates - gates
                        
                    #         # 2. Create the zero array for padding:
                    #         #    (padding_needed, 12) shape
                    #         zero_padding = np.zeros((padding_needed, 12), dtype=target.dtype)
                        
                    #         # 3. Concatenate the original target and the zero padding along the vertical axis (axis=0)
                    #         target = np.concatenate([target, zero_padding], axis=0)

        
        # Instead of creating a single giant NumPy array, keep them as a list.
        # This works perfectly with different image resolutions.
        self.images = all_clean_images
        self.targets = all_clean_targets

        # We also need to convert targets to a single array for consistency.
        # Since all targets are now (12,) or (1, 12), this will work.
        self.targets = np.array(self.targets)

        print(f"Total valid samples to be used for training: {len(self.images)}")


    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # --- The New, Simplified __getitem__ ---
        # No padding or masking needed!
        
        image = self.images[idx]
        target = self.targets[idx]
        
        # The target is guaranteed to be for one gate, so we just flatten it.
        target_tensor = torch.from_numpy(target).float().flatten()
            
        image_tensor = torch.from_numpy(image).float() / 255.0
        
        if self.transform:
            image_tensor = self.transform(image_tensor)
            
        # Return only the image and target. No mask.
        return image_tensor, target_tensor

In [6]:
# 1. Specify file paths
DATA_ROOT = '/kaggle/input/ae4353-y25' # Corrected path from your code

# Training on all files
#h5_paths = [os.path.join(DATA_ROOT, fname) for fname in os.listdir(DATA_ROOT) if fname.endswith('.h5')]

files_to_use = [
        'BaltimoreMobile.h5', 'LittletonBlue.h5', 'Warsaw1.h5',
        'autonomous_flight-01a-ellipse.h5', 'autonomous_flight-13a-trackRATM.h5'
    ]
h5_paths = [os.path.join(DATA_ROOT, fname) for fname in files_to_use]


# 2. Define transforms
TARGET_RESOLUTION = (224, 224)
data_transforms = transforms.Compose([
    transforms.Resize(TARGET_RESOLUTION),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Create ONE instance of our simple, single-gate dataset
full_dataset = H5DroneDataset(
    h5_paths=h5_paths,
    transform=data_transforms
)

# 4. Create your training and validation split
total_len = len(full_dataset)
train_len = int(0.85 * total_len)
val_len = total_len - train_len
train_set, val_set = torch.utils.data.random_split(full_dataset, [train_len, val_len])


# 5. Create train and validation loaders (NO collate_fn needed)
train_loader = DataLoader(
    dataset=train_set,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    dataset=val_set,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

# 6. Test the DataLoader (it will return two items)
print("\nFetching one batch from the DataLoader...")
images_batch, targets_batch = next(iter(train_loader))
print("SUCCESS! The batch was loaded with uniform, single-gate targets.")
print(f"Images batch shape:  {images_batch.shape}")
print(f"Targets batch shape: {targets_batch.shape}") # Should be [32, 12]

Scanning all files to filter out multi gate images...


Scanning Files: 100%|██████████| 5/5 [00:23<00:00,  4.78s/it]

Total valid samples to be used for training: 731

Fetching one batch from the DataLoader...
SUCCESS! The batch was loaded with uniform, single-gate targets.
Images batch shape:  torch.Size([32, 3, 224, 224])
Targets batch shape: torch.Size([32, 12])


In [7]:
class VanillaCNN(nn.Module):
    """
    A vanilla convolutional neural network model.
    with the following architecture:
    - Three convolutional layers each with 3x3 kernels, stride=1 w/ ReLU activation followed by a max pooling layer with 2x2 kernels.
    - The first layer has 16 output channels, the second has 32 output channels, and the third has 64 output channels.
    - Flatten the output of the last convolutional layer.
    - Two fully connected layers with ReLU activation with 128 hidden units and 64 hidden units.
    - One fully connected layer with linear activation as the output layer.

    Args:
        channel_in (int): Number of input channels.
        channel_out (int): Number of output channels.
        last_layer_bias (bool, optional): Whether to include bias in the last layer. Defaults to True.

    Attributes:
        conv1 (nn.Conv2d): First convolutional layer.
        conv2 (nn.Conv2d): Second convolutional layer.
        conv3 (nn.Conv2d): Third convolutional layer.
        fc1 (nn.LazyLinear): First fully connected layer.
        fc2 (nn.Linear): Second fully connected layer.
        fc3 (nn.Linear): Third fully connected layer.

    Methods:
        forward(x): Forward pass of the model.

    """
    
    def __init__(self, channel_in, channel_out, last_layer_bias=True):
        super().__init__()

        #--------------------------------------------------------------
        # TODO 2.1: Define the layers of the CNN as described above
        #--------------------------------------------------------------

        self.conv1 = nn.Conv2d(channel_in, 16, kernel_size=3, stride=1)     #stride is hoe die over de pixels heengaan
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1)
        
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=1)
        
        self.fc1 = nn.LazyLinear(out_features=128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, channel_out, bias=last_layer_bias)

        
    def forward(self, x):
        

        x = F.relu(self.conv1(x))       # Use F.relu for space efficent coding
        x = F.max_pool2d(x, 2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
    
        return self.fc3(x)
        

In [8]:
def custom_mse_loss(pred, target):
    """
    Calculates the MSE loss only on the (x, y) coordinates.

    Args:
        pred (torch.Tensor): The model's prediction tensor with shape (N, 8).
        target (torch.Tensor): The ground-truth tensor with shape (N, 12).
                               It contains (x, y, flag) for 4 corners.

    Returns:
        torch.Tensor: A single scalar value representing the loss.
    """
    # 1. Reshape the target to group by corner: (N, 12) -> (N, 4, 3)
    target_reshaped = target.view(-1, 4, 3)

    # 2. Slice to get only the x and y coordinates: (N, 4, 3) -> (N, 4, 2)
    target_coords = target_reshaped[:, :, :2]

    # 3. Flatten to match the prediction shape: (N, 4, 2) -> (N, 8)
    target_coords = target_coords.flatten(start_dim=1)

    # 4. Calculate the MSE loss using the filtered target
    loss = criterion(pred, target_coords)
    
    return loss

In [9]:
def evaluate(val_loader, model, criterion):
    """
    Evaluate the performance of a model on a given dataloader.
    """
    model.eval()
    
    losses = [] # Renamed to 'losses' to be more descriptive

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Eval", leave=False):
            images, targets = batch

            images = images.to(device) # Plural 'images' matches batch
            targets = targets.to(device) # Plural 'targets' matches batch
            
            # --- FIX 1: Typo Correction ---
            # 'prediciton' was misspelled.
            prediction = model(images)
            
            # --- FIX 2: Correctly calculate the loss item ---
            loss_item = custom_mse_loss(prediction, targets) # 'targets' matches variable from batch
            
            # --- FIX 3: Correct Syntax for Appending ---
            # You must get the scalar value with .item() BEFORE appending to the list.
            # The .append() method itself returns None.
            losses.append(loss_item.item())

    return losses

In [10]:
def train_epoch(train_loader, val_loader, model, criterion, optimizer):
    """
    Trains the model for one epoch using the provided data loaders, model, optimizer, and criterion.
    Args:
        train_loader (torch.utils.data.DataLoader): Data loader for the training set.
        val_loader (torch.utils.data.DataLoader): Data loader for the validation set.
        model (torch.nn.Module): The model to be trained.
        optimizer (torch.optim.Optimizer): The optimizer used for training.
        criterion (torch.nn.Module): The loss function used for training.
    Returns:
        tuple: A tuple containing the training loss, training performance, validation performance, and validation predictions.
    """
    # --------------------------------------------------------------
    # TODO 4.1: Implement the training loop for one epoch
    # --------------------------------------------------------------
    
    model.train()

    batches = tqdm(train_loader, desc='training', leave=False)

    for input, vector, angle in batches:

        input_new = input.to(device)
        vector_new = vector.to(device).float()
        angle = angle.to(device)

        optimizer.zero_grad()

        output = model(input_new)

        loss = criterion(output, vector_new)
        loss.backward()
        
        optimizer.step()

        train_loss, train_performance, _ = evaluate(train_loader, model, criterion)
    _, val_performance, val_pred = evaluate(val_loader, model, criterion)

    # --------------------------------------------------------------
    # END OF TODO 4.1
    # --------------------------------------------------------------

    return train_loss, train_performance, val_performance, val_pred

In [11]:
#model = VanillaCNN(channel_in=3, channel_out)


In [12]:
!pip install hdf5plugin opencv-python-headless moviepy tqdm 

In [13]:
# def h5_to_video(h5, output_dir, fps):
#     output_dir.mkdir(parents=True, exist_ok=True)

#     # get images and targets
#     h5f = h5py.File(h5, "r")
#     images = h5f["images"]
#     targets = [h5f[f"targets/{i:05d}"][()] for i in range(len(images))]

#     # draw targets on images
#     frames = []
#     for image, target in tqdm(zip(images, targets), total=len(images)):
#         image = image.transpose(1, 2, 0)
#         frame = image.copy()
#         for gate in target:
#             xy = gate.reshape(-1, 3)[..., :2] * image.shape[1::-1]
#             visibility = gate.reshape(-1, 3)[..., 2]
#             if np.all(visibility > 0):
#                 cv2.polylines(frame, [xy.astype(int)], isClosed=True, color=(0, 255, 0), thickness=2)
#             else:
#                 # Draw lines between visible corners
#                 for i in range(len(xy) - 1):
#                     if visibility[i] > 0 and visibility[i + 1] > 0:
#                         cv2.line(frame, tuple(xy[i].astype(int)), tuple(xy[i + 1].astype(int)), color=(0, 255, 0), thickness=2)
#                 # Draw line from the last corner to the first to form a loop if both are visible
#                 if visibility[-1] > 0 and visibility[0] > 0:
#                     cv2.line(frame, tuple(xy[-1].astype(int)), tuple(xy[0].astype(int)), color=(0, 255, 0), thickness=2)
#         frames.append(frame)

#     h5f.close()

#     clip = ImageSequenceClip(frames, fps=fps)
#     clip.write_videofile(str(output_dir / f"{h5.stem}.mp4"), codec="libx264")

In [14]:
# h5_to_video(Path("/kaggle/input/ae4353-y25/WashingtonOBNewLight.h5"), Path('.'), 30)